# ARCHITECTURE 1 (Ensemble Baseline)

In [5]:
import pandas as pd

path= "../data/processed/combined_hybrid_student_reddit_data.csv"
df = pd.read_csv(path)

df.head()


,clean_text,binary_label,source
0,"at this hour i just woke up, even though yeste...",0,Reddit
1,"A 22-year-old Male student with CGPA 3.39, 4.0...",1,Student
2,"A 27-year-old Female student with CGPA 2.94, 5...",1,Student
3,"A 28-year-old Female student with CGPA 3.44, 1...",0,Student
4,"A 32-year-old Female student with CGPA 2.92, 4...",1,Student


In [6]:
X = df["clean_text"]
y = df["binary_label"].astype(int)

##### RANDOM FOREST PIPELINE

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Pipeline
rf_pipeline = Pipeline(steps=[("tfidf", TfidfVectorizer(max_features=3000,ngram_range=(1, 2),stop_words="english",min_df=10,max_df=0.8)),
                             ("rf", RandomForestClassifier(n_estimators=100,max_depth=20,random_state=42,class_weight="balanced",
                              min_samples_split=10,min_samples_leaf=5,max_features='sqrt',bootstrap=True,))])

# Train
rf_pipeline.fit(X_train, y_train)

# Predict
y_pred_rf = rf_pipeline.predict(X_test)

# Evaluate
print("Random Forest Baseline Results")
print(classification_report(y_test, y_pred_rf))
print("Accuracy:", accuracy_score(y_test, y_pred_rf))


Random Forest Baseline Results
              precision    recall  f1-score   support

           0       0.64      0.84      0.73      5581
           1       0.87      0.70      0.77      8477

    accuracy                           0.75     14058
   macro avg       0.76      0.77      0.75     14058
weighted avg       0.78      0.75      0.75     14058

Accuracy: 0.7522407170294494


In [8]:
from xgboost import XGBClassifier

xgb_pipeline = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(max_features=3000,ngram_range=(1, 2),stop_words="english",min_df=10,max_df=0.8)),
    ("xgb", XGBClassifier(n_estimators=150,max_depth=6,learning_rate=0.1,subsample=0.8,colsample_bytree=0.8,
                          eval_metric="logloss",random_state=42,n_jobs=-1,tree_method='hist',verbosity=1))])

# Train
xgb_pipeline.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb_pipeline.predict(X_test)

# Evaluate
print("XGBoost Baseline Results")
print(classification_report(y_test, y_pred_xgb))
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))


XGBoost Baseline Results
              precision    recall  f1-score   support

           0       0.84      0.65      0.73      5581
           1       0.80      0.92      0.85      8477

    accuracy                           0.81     14058
   macro avg       0.82      0.78      0.79     14058
weighted avg       0.81      0.81      0.80     14058

Accuracy: 0.8105704936690852


In [9]:
# save the models 
import joblib

joblib.dump(xgb_pipeline, "../models/baseline_xgb_model.pkl")
joblib.dump(rf_pipeline, "../models/baseline_rf_model.pkl")


['../models/baseline_rf_model.pkl']

In [10]:
#import sys
#!{sys.executable} -m pip install xgboost


[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: C:\Users\NUGGET\mental_health_prediction\venv\Scripts\python.exe -m pip install --upgrade pip
